In [134]:
!pip install xgboost optuna --quiet

In [135]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import warnings, os
from pathlib import Path
import yfinance as yf

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

In [136]:
np.random.seed(20160101)
os.makedirs("results", exist_ok=True)

# SECTION 1 — UNIVERSE & DATA

 We download the full historical price series for every ticker that has ever
 been in the Dow 30 across our study period. This is the "warehouse" — it
 holds all prices but does NOT determine which stocks are tradeable on any
 given date. That is handled by get_constituents_on_date() below.

 Why this matters: if we used today's Dow 30 for all historical dates, we
 would be training on AMZN, NVDA, and SHW going back to 2016 — stocks that
 were added much later. Winners that get added to an index look great in
 hindsight. This is survivorship bias and it inflates backtest returns.


In [137]:

# The 30 constituents as of April 2016 (our start date)
_BASELINE_APR2016 = {
    "AAPL", "AXP", "BA",  "CAT",  "CSCO", "CVX",  "DD",
    "DIS",  "GE",  "GS",  "HD",   "IBM",  "INTC", "JNJ",
    "JPM",  "KO",  "MCD", "MMM",  "MRK",  "MSFT", "NKE",
    "PFE",  "PG",  "RTX", "TRV",  "UNH",  "V",    "VZ",
    "WMT",  "XOM",
}

# Every constituent change since April 2016
# Format: (effective_date, [added], [removed])
_CHANGES = [
    ("2018-06-26", ["WBA"],               ["GE"]),
    ("2019-04-02", ["DOW"],               ["DD"]),
    ("2020-04-06", ["RTX"],               ["UTX"]),
    ("2020-08-31", ["AMGN", "CRM", "HON"],["XOM", "PFE", "RTX"]),
    ("2024-02-26", ["AMZN", "SHW"],       ["WBA", "INTC"]),
    ("2024-11-01", ["NVDA"],              ["DOW"]),
]

# Hard-coded data split boundaries
# These are chosen to ensure each split contains a distinct market regime:
#   Train  : 2016–2020  bull market + 2018 Q4 selloff
#   Val    : 2020–2022  COVID crash, recovery, rate-hike onset
#   Test   : 2022–now   2022 bear market, 2023-2025 bull run
TRAIN_CUTOFF = pd.Timestamp("2020-01-01")
VAL_CUTOFF   = pd.Timestamp("2022-01-01")

# Optuna trials
N_OPTUNA_TRIALS = 80


def get_constituents_on_date(date: pd.Timestamp) -> set:
    """
    Return the exact set of Dow 30 tickers that were in the index on `date`.

    Replays every addition and removal in _CHANGES up to and including `date`.
    WBA and UTX are permanently excluded: WBA has persistent data quality
    issues; UTX was renamed to RTX before our study period begins.

    This function is the single source of truth for the point-in-time (PIT)
    universe. It is called on every date in both feature engineering and the
    backtest loop — never replaced by a static global list.
    """
    constituents = set(_BASELINE_APR2016)
    for change_date_str, added, removed in _CHANGES:
        if date >= pd.Timestamp(change_date_str):
            constituents.update(added)
            constituents -= set(removed)
    constituents -= {"WBA", "UTX"}
    return constituents


def fetch_prices(
    start_date: str = "2016-04-01",
    end_date:   str = "2026-04-18",
) -> pd.DataFrame:
    """
    Download daily adjusted close prices for all historical Dow constituents.

    Returns a single DataFrame where columns are tickers and rows are dates.
    Forward-fills up to 10 days to handle trading holidays. Drops any ticker
    with no data at all.

    Note: this downloads MORE tickers than are in the index at any one time.
    The extra tickers are needed so we can look up prices for stocks that
    joined the index partway through the study period. get_constituents_on_date()
    controls which subset is actually used on each date.
    """
    all_tickers = set(_BASELINE_APR2016)
    for _, added, removed in _CHANGES:
        all_tickers.update(added)
    all_tickers -= {"WBA", "UTX"}
    all_tickers  = sorted(all_tickers)

    raw    = yf.download(all_tickers, start=start_date, end=end_date,
                         auto_adjust=True, progress=False)
    prices = raw["Close"].ffill(limit=10)
    prices = prices.dropna(axis=1, how="all")
    return prices

### Feature Engineering

 Three features are computed for each stock on each date:

   mom_12_1 : 12-month return skipping the last month. The skip avoids the
              well-documented short-term reversal effect (1-month mean reversion).
              Lookback = 252 trading days back, skip = 21 trading days.

   vol_60   : 60-day realised annualised volatility. Used as a risk signal —
              high-vol stocks tend to have weaker risk-adjusted momentum.

   rsi      : Relative Strength Index over a tunable period (default 14 days).
              Values above 70 = potentially overbought; below 30 = oversold.

 LOOK-AHEAD BIAS SAFEGUARDS:
   - Only prices[0..di] are used when computing features for date di.
     No price at di+1 or later is touched.
   - fwd_ret (the 5-day forward return) is computed here purely as a
     TRAINING LABEL. It is never used as a feature or passed to the live
     signal computation. During backtesting, predict_scores() uses only
     mom_12_1, vol_60, and rsi — none of which require future prices.
   - The universe on each date is determined by get_constituents_on_date(date),
     not by the full column list of the prices DataFrame.

In [138]:
FEATURE_COLS = ["mom_12_1", "vol_60", "rsi"]

In [139]:
def compute_rsi(prices_arr: np.ndarray, period: int) -> float:
    """
    Compute RSI from the last (period + 1) prices in prices_arr.

    Uses only prices_arr[-(period+1):] so it never looks beyond the current
    date. Returns 100.0 if there are no down days (avoid divide-by-zero).
    """
    delta  = np.diff(prices_arr[-(period + 1):])
    gains  = delta[delta > 0].sum() / period
    losses = -delta[delta < 0].sum() / period
    if losses == 0:
        return 100.0
    return 100.0 - 100.0 / (1.0 + gains / losses)

In [140]:
def compute_features(
    prices:      pd.DataFrame,
    lookback:    int = 252,   # trading days for 12-1 momentum (~12 months)
    vol_window:  int = 60,    # trading days for realised vol
    rsi_period:  int = 14,    # RSI lookback period
    skip:        int = 21,    # skip last month to avoid short-term reversal
    fwd_days:    int = 5,     # forward return horizon (for training label only)
) -> pd.DataFrame:
    """
    Build a panel of (date, ticker, features, label) for every valid date.

    The loop advances one day at a time. On each date:
      1. The PIT universe is determined — only stocks in the index that day.
      2. All three features are computed using only past prices (no lookahead).
      3. The forward return is computed as the label for XGBoost training.
      4. fwd_rank — the cross-sectional percentile rank of fwd_ret — is added
         as the actual training target (rank regression is more stable than
         raw return regression across different market regimes).

    The loop stops fwd_days before the end of the price series so that forward
    returns can always be computed without running off the end of the array.

    Rows with NaN in any feature or label are dropped. Dates where fewer than
    5 stocks have valid data are discarded entirely.
    """
    log_rets = np.log(prices / prices.shift(1))
    min_day  = lookback + skip + rsi_period + 1
    records  = []

    for di in range(min_day, len(prices) - fwd_days):
        date   = prices.index[di]
        fwd_di = di + fwd_days   # index of the price fwd_days later

        # Step 1: point-in-time universe for this exact date
        pit_tickers = get_constituents_on_date(date)
        pit_tickers = [t for t in pit_tickers if t in prices.columns]

        row_data = []
        for t in pit_tickers:
            px = prices[t].values

            # Guard: skip if we don't have enough history yet, or price is NaN
            if di - lookback - skip < 0 or np.isnan(px[di]):
                continue

            # Feature 1: 12-1 momentum (no future prices used)
            mom = px[di - skip] / px[di - lookback - skip] - 1.0

            # Feature 2: 60d realised vol (no future prices used)
            r_slice = log_rets[t].iloc[di - vol_window: di].values
            if np.any(np.isnan(r_slice)):
                continue
            vol = float(np.std(r_slice)) * np.sqrt(252.0)

            # Feature 3: RSI (no future prices used — passes only px[0..di])
            rsi = compute_rsi(px[: di + 1], rsi_period)

            # Training label only — NEVER used as a feature
            fwd = px[fwd_di] / px[di] - 1.0

            row_data.append({
                "date":     date,
                "ticker":   t,
                "mom_12_1": mom,
                "vol_60":   vol,
                "rsi":      rsi,
                "fwd_ret":  fwd,
            })

        if not row_data:
            continue

        df_row = pd.DataFrame(row_data)

        # Cross-sectional percentile rank (0–1) — training target for XGBoost
        # na_option="keep" propagates NaN rather than silently ranking them
        df_row["fwd_rank"] = df_row["fwd_ret"].rank(pct=True, na_option="keep")

        # Drop any row with NaN in features or label
        df_row = df_row.dropna(subset=FEATURE_COLS + ["fwd_ret", "fwd_rank"])

        if len(df_row) >= 5:   # need at least 5 stocks for a meaningful rank
            records.append(df_row)

    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

### Regime Detection
Primary issue with momentum based trading is when the momentum stops... bad.

Unless you've got a backstop in to change course on a crash, the trading strategy will do very poorly.


 Momentum strategies are known to suffer badly during sudden market reversals.
 The classic example is the COVID crash in Feb-Mar 2020: the highest-momentum
 stocks (recent winners) fell hardest and fastest when the crash hit.

 We classify each day into one of three regimes using two signals computed
 on a point-in-time equal-weight market index:

   - roll_vol : rolling 20-day annualised volatility of the market index
   - roll_dd  : rolling 60-day drawdown from peak

 State logic (crash overrides volatile):
   - crash    : roll_vol > 0.28  OR  roll_dd < -0.13  → go 100% cash
   - volatile : roll_vol > 0.17  OR  roll_dd < -0.07  → scale to 70% exposure
   - trending : otherwise                              → full exposure

 LOOK-AHEAD BIAS SAFEGUARD:
   The market index is built using get_constituents_on_date(date) on every
   date. This ensures that the regime signal on any given day only reflects
   stocks that were actually in the index on that day — not stocks added
   later that happened to have performed well.

 Expected crash periods in a 2016–2026 backtest:
   - Q4 2018  (Fed rate fears, -20% from peak)
   - Feb-Mar 2020  (COVID crash, -34% from peak)
   - Jan-Oct 2022  (rate hike cycle, -22% from peak)

In [141]:
def classify_regimes(
    prices:     pd.DataFrame,
    vol_window: int   = 20,    # rolling window for vol estimate (trading days)
    dd_window:  int   = 60,    # rolling window for drawdown (trading days)
    vol_crash:  float = 0.28,  # annualised vol → crash
    vol_vol:    float = 0.17,  # annualised vol → volatile
    dd_crash:   float = -0.13, # drawdown from peak → crash
    dd_vol:     float = -0.07, # drawdown from peak → volatile
) -> pd.Series:
    """
    Label each trading day as 'trending', 'volatile', or 'crash'.

    Builds a daily PIT equal-weight index then computes rolling vol and
    drawdown on it. Both signals are checked independently — whichever is
    more severe determines the state (crash > volatile > trending).
    """
    # Build PIT equal-weight index: only average stocks in index on each date
    pit_index = []
    for date in prices.index:
        members = get_constituents_on_date(date)
        valid   = [t for t in members
                   if t in prices.columns and pd.notna(prices.loc[date, t])]
        pit_index.append(prices.loc[date, valid].mean() if valid else np.nan)

    mkt      = pd.Series(pit_index, index=prices.index).ffill()
    log_rets = np.log(mkt / mkt.shift(1))
    roll_vol = log_rets.rolling(vol_window).std() * np.sqrt(252)
    roll_pk  = mkt.rolling(dd_window).max()
    roll_dd  = (mkt - roll_pk) / roll_pk

    regimes = pd.Series("trending", index=prices.index, dtype=str)
    regimes[(roll_vol > vol_vol)   | (roll_dd < dd_vol)]   = "volatile"
    regimes[(roll_vol > vol_crash) | (roll_dd < dd_crash)] = "crash"   # overrides volatile
    return regimes

### Building your Portfolio
#### XGBOOST CROSS-SECTIONAL RANKER


 XGBoost is trained to predict each stock's forward-return rank within its
 cross-sectional peer group (fwd_rank, a percentile from 0 to 1). A stock
 scoring near 1.0 is predicted to be a top-quartile performer over the next
 5 days; a stock scoring near 0.0 is predicted to be a bottom-quartile
 underperformer.

 We use regression on the rank percentile rather than raw return prediction
 because:
   - Rank is bounded [0,1] — more stable across different market regimes
   - Rank removes the common market factor (everyone going up or down together)
   - The portfolio optimiser cares about relative ordering, not absolute levels

 LOOK-AHEAD BIAS SAFEGUARD:
   train_xgboost() receives ONLY rows whose date falls in the train split
   (before TRAIN_CUTOFF = 2020-01-01). It never sees val or test dates.
   The model is fitted once and then held fixed for the entire val + test period.
   In a production system you would retrain on a rolling window, but a fixed
   model is the correct approach for an honest single backtest evaluation.

In [142]:
def train_xgboost(train_df: pd.DataFrame) -> xgb.XGBRegressor:
    """
    Train XGBoost on the training split to predict cross-sectional rank.

    Cleans the training data by replacing inf with NaN and dropping any row
    with NaN in features or target. Raises if the cleaned dataset is empty.
    """
    clean = (train_df[FEATURE_COLS + ["fwd_rank"]]
             .replace([np.inf, -np.inf], np.nan)
             .dropna())

    X = clean[FEATURE_COLS].values.astype(np.float32)
    y = clean["fwd_rank"].values.astype(np.float32)

    if len(X) == 0:
        raise ValueError(
            "Training data is empty after cleaning. "
            "Check compute_features() output for the train split."
        )

    model = xgb.XGBRegressor(
        n_estimators     = 300,
        max_depth        = 3,     # shallow trees reduce overfitting
        learning_rate    = 0.05,
        subsample        = 0.8,   # row subsampling per tree
        colsample_bytree = 1.0,   # use all features (only 3, so no reduction needed)
        min_child_weight = 5,     # minimum samples per leaf
        reg_lambda       = 1.0,   # L2 regularisation on leaf weights
        reg_alpha        = 0.1,   # L1 regularisation
        objective        = "reg:squarederror",
        eval_metric      = "rmse",
        random_state     = 42,
        n_jobs           = -1,
        verbosity        = 0,
    )
    model.fit(X, y)
    return model


def predict_scores(model: xgb.XGBRegressor, feat_df: pd.DataFrame) -> np.ndarray:
    """
    Score a snapshot of stocks using the trained XGBoost model.

    feat_df must contain FEATURE_COLS columns. Only past prices are used to
    build feat_df in the backtest loop, so this call is look-ahead free.
    """
    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    return model.predict(X)

 # SECTION 5 — RIDGE PORTFOLIO OPTIMISATION

 Given XGBoost scores for each stock, Ridge optimisation sizes positions
 proportionally to their score while penalising excessive concentration.

 - Objective:  maximise  Σ score_i * w_i  −  λ * ||w||²
 - Subject to: Σ w_i = 1,  w_i ≥ 0  (long-only)

 - Closed-form solution:  w_i* ∝ max(0, score_i) / (2λ)
 Then project to the probability simplex (normalise to sum = 1).

 - Higher λ → weights converge toward equal-weight across all top-N stocks
 - Lower  λ → weights concentrate on the highest-scoring stocks

 POSITION CAP:
 A hard cap of max_weight (default 15%) per position is enforced after the
 Ridge solution. This prevents degenerate outcomes when λ is very small and
 one stock dominates (as seen with the 3-stock, 70%-concentrated portfolio
 during earlier debugging). The cap iteratively redistributes excess weight
 to uncapped positions until all weights are below the threshold.

In [143]:
def ridge_optimize(
    scores:       np.ndarray,
    tickers:      list,
    ridge_lambda: float = 0.10,
    top_n:        int   = 7,
    max_weight:   float = 0.15,   # hard cap per position
) -> dict:
    """
    Build a Ridge-regularised long-only portfolio from XGBoost scores.

    Selects the top_n stocks by score, computes analytical Ridge weights,
    applies a per-position cap, then renormalises to sum = 1.
    """
    # Select top-N by predicted rank score
    ranked_idx       = np.argsort(scores)[::-1][:top_n]
    selected_tickers = [tickers[i] for i in ranked_idx]
    selected_scores  = scores[ranked_idx]

    # Analytical Ridge weights (unconstrained)
    raw_w = np.maximum(0.0, selected_scores) / (2.0 * ridge_lambda)
    total = raw_w.sum()
    w     = raw_w / total if total > 0 else np.full(top_n, 1.0 / top_n)

    # Iterative position cap: redistribute weight from capped positions
    for _ in range(20):
        breached = w > max_weight
        if not breached.any():
            break
        excess   = (w[breached] - max_weight).sum()
        w[breached] = max_weight
        uncapped = ~breached
        if uncapped.any():
            # Redistribute excess proportionally to uncapped positions
            w[uncapped] += excess * (w[uncapped] / w[uncapped].sum())

    # Final clip and renormalise to handle floating-point edge cases
    w = np.clip(w, 0.0, max_weight)
    w /= w.sum()

    return dict(zip(selected_tickers, w))

### BackTest the Thing
 The backtest simulates daily portfolio management from the start of the
 evaluation period (val or test) to the end of the price history.

 On each trading day:
   1. Determine today's PIT universe (stocks actually in the index today)
   2. Compute daily returns for each stock in the PIT universe
   3. Check the regime label for today
   4. If a rebalance is due (every rebal_freq weeks):
        a. Pull the most recent feature snapshot available (≤ today)
        b. Score PIT stocks using the frozen XGBoost model
        c. Apply regime scaling:
             crash    → weights = {} (100% cash, kill-switch fires)
             volatile → build portfolio then scale all weights × 0.70
             trending → build portfolio at full weight
        d. Build Ridge-optimised weights
   5. Compute strategy P&L from weights × daily returns
   6. Compute benchmark P&L as equal-weight average of PIT universe returns

 LOOK-AHEAD BIAS SAFEGUARDS:
   - The XGBoost model was trained only on dates before TRAIN_CUTOFF.
     It has never seen any data from the val or test period.
   - When looking up features at rebalance time, we use:
       snap_date = test_df[test_df["date"] <= date]["date"].max()
     This takes the MOST RECENT feature row that is STRICTLY ≤ today.
     It never uses today's or tomorrow's features.
   - Regime labels are computed on the full price history before the backtest
     begins. This is acceptable because regime detection uses rolling statistics
     (vol, drawdown) that are themselves look-ahead free — each day's regime
     label only depends on prices up to and including that day.
   - The benchmark uses the same PIT universe as the strategy. It does NOT
     average all columns in the prices DataFrame (which would include AMZN
     and NVDA before they joined the index).

 The `mode` parameter controls which date window is used as the eval period:
   mode="val"  → evaluate on val split  (used by Optuna during tuning)
   mode="test" → evaluate on test split (used for final reported metrics only)

In [144]:
def run_backtest(
    prices:       pd.DataFrame,
    features_df:  pd.DataFrame,
    regimes:      pd.Series,
    rsi_period:   int   = 14,
    rebal_freq:   int   = 4,      # rebalance every N weeks
    ridge_lambda: float = 0.10,
    top_n:        int   = 7,
    mode:         str   = "val",  # "val" for Optuna, "test" for final evaluation
) -> dict:
    """
    Walk-forward backtest over the val or test split.

    Returns a dict of performance metrics and curves. Returns sentinel values
    (sharpe=-99) if there is insufficient data to run.
    """
    rebal_days   = rebal_freq * 5
    unique_dates = np.sort(features_df["date"].unique())

    # Hard date-based split — same boundaries used everywhere
    train_dates = unique_dates[unique_dates <  TRAIN_CUTOFF]
    val_dates   = unique_dates[(unique_dates >= TRAIN_CUTOFF) &
                               (unique_dates <  VAL_CUTOFF)]
    test_dates  = unique_dates[unique_dates >= VAL_CUTOFF]
    eval_dates  = val_dates if mode == "val" else test_dates

    if len(train_dates) < 20 or len(eval_dates) < 20:
        return {"sharpe": -99.0, "cagr": -99.0, "max_dd": -99.0}

    # Train XGBoost on training split only — never touches val or test dates
    train_df = features_df[features_df["date"].isin(train_dates)]
    model    = train_xgboost(train_df)

    # Feature rows available during the evaluation period
    eval_df    = features_df[features_df["date"].isin(eval_dates)]
    eval_start = pd.Timestamp(eval_dates[0])
    eval_prices = prices[prices.index >= eval_start].copy()

    # Initialise portfolio state
    weights    = {}            # empty until first rebalance
    strat_val  = 100.0
    bench_val  = 100.0
    peak       = 100.0
    max_dd     = 0.0
    kills      = 0
    last_rebal = -rebal_days   # negative forces rebalance on day 1

    strat_curve = []
    bench_curve = []
    date_index  = []
    regime_log  = []
    all_weights_log = [] # Store weights at each rebalance for analysis

    for di in range(1, len(eval_prices)):
        date = eval_prices.index[di]

        # PIT universe: only stocks in the index on this specific date
        pit_tickers = get_constituents_on_date(date)
        pit_tickers = [t for t in pit_tickers if t in eval_prices.columns]

        # Daily returns for PIT stocks only (no future prices)
        daily_rets = {}
        for t in pit_tickers:
            p0 = eval_prices[t].iloc[di - 1]
            p1 = eval_prices[t].iloc[di]
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                daily_rets[t] = p1 / p0 - 1.0

        reg = regimes.loc[date] if date in regimes.index else "trending"
        regime_log.append(reg)

        # Rebalance logic
        if di - last_rebal >= rebal_days:
            last_rebal = di

            # Most recent feature snapshot strictly ≤ today (no lookahead)
            avail = eval_df[eval_df["date"] <= date]
            if not avail.empty:
                snap_date = avail["date"].max()
                snap = (
                    eval_df[
                        (eval_df["date"] == snap_date) &
                        (eval_df["ticker"].isin(pit_tickers))
                    ]
                    .set_index("ticker")
                    .dropna(subset=FEATURE_COLS)
                )

                if len(snap) >= top_n:
                    scores_arr    = predict_scores(model, snap.reset_index())
                    tickers_avail = snap.index.tolist()

                    if reg == "crash":
                        # Kill-switch: liquidate everything → 100% cash
                        weights = {}
                        kills  += 1
                    else:
                        # Scale gross exposure based on regime
                        scale = 0.70 if reg == "volatile" else 1.00
                        opt_w = ridge_optimize(
                            scores_arr, tickers_avail, ridge_lambda, top_n
                        )
                        weights = {t: w * scale for t, w in opt_w.items()}

                    # Store weights for analysis
                    all_weights_log.append({
                        "date": snap_date, # Use snap_date as this is when the portfolio was formed
                        "weights": weights.copy() # Store a copy of the weights
                    })

        # Strategy daily P&L: weights × realised returns
        strat_ret = sum(
            weights.get(t, 0.0) * daily_rets.get(t, 0.0)
            for t in weights
        )

        # Benchmark: equal-weight average of PIT universe returns (same universe
        # as strategy — no AMZN/NVDA before they were added to the index)
        bench_ret = (
            float(np.mean([daily_rets[t] for t in pit_tickers if t in daily_rets]))
            if any(t in daily_rets for t in pit_tickers) else 0.0
        )

        strat_val *= (1.0 + strat_ret)
        bench_val *= (1.0 + bench_ret)
        peak       = max(peak, strat_val)
        max_dd     = min(max_dd, (strat_val - peak) / peak)

        strat_curve.append(strat_val)
        bench_curve.append(bench_val)
        date_index.append(date)

    if len(strat_curve) < 50:
        return {"sharpe": -99.0, "cagr": -99.0, "max_dd": -99.0}

    # Performance metrics
    daily_rets_arr = np.diff(strat_curve) / np.array(strat_curve[:-1])
    n_years        = len(strat_curve) / 252.0
    cagr           = (strat_val / 100.0) ** (1.0 / n_years) - 1.0
    bench_cagr     = (bench_val / 100.0) ** (1.0 / n_years) - 1.0
    sharpe         = (
        (daily_rets_arr.mean() / daily_rets_arr.std()) * np.sqrt(252.0)
        if daily_rets_arr.std() > 0 else 0.0
    )
    feat_imp = dict(zip(FEATURE_COLS, model.feature_importances_))

    return {
        "sharpe":      round(float(sharpe),          4),
        "cagr":        round(float(cagr * 100),      2),
        "bench_cagr":  round(float(bench_cagr * 100), 2),
        "max_dd":      round(float(max_dd * 100),    2),
        "cum_ret":     round(strat_val - 100.0,      2),
        "kills":       kills,
        "strat_curve": strat_curve,
        "bench_curve": bench_curve,
        "date_index":  date_index,
        "regime_log":  regime_log,
        "feat_imp":    feat_imp,
        "model":       model,
        "weights":     weights,
        "all_weights_log": all_weights_log, # Return the log of all weights
    }

### Param Optimization

 Four hyperparameters are tuned using Optuna's TPE (Tree-structured Parzen
 Estimator) sampler — a Bayesian search method that learns which regions of
 the parameter space produce good results and focuses sampling there.

   - rsi_period   : [7, 10, 14, 21, 28] days
   - rebal_freq   : [1, 2, 4, 6, 8, 12] weeks
   - ridge_lambda : log-uniform [0.10, 2.0]  ← floor at 0.10 to prevent
                  degenerate near-zero-lambda portfolios
   - top_n        : [3, 5, 7, 9, 12] stocks

 LOOK-AHEAD BIAS SAFEGUARD:
   Every Optuna trial calls run_backtest(..., mode="val"). This evaluates
   only on the validation split (2020–2022). The test split (2022–present)
   is completely invisible to the optimiser. Sharpe on the test split is
   reported ONLY once, after the best parameters have been chosen.

 CACHING:
   compute_features() is expensive (iterates over every date × ticker).
   We cache the result per rsi_period so that multiple Optuna trials that
   share the same rsi_period don't recompute features from scratch.

In [145]:
def run_optuna(
    prices:   pd.DataFrame,
    regimes:  pd.Series,
    n_trials: int = N_OPTUNA_TRIALS,
    seed:     int = 42,
) -> tuple:
    """
    Run Optuna TPE hyperparameter search on the validation split.

    Returns (study, best_params).
    """
    feat_cache: dict = {}

    def objective(trial: optuna.Trial) -> float:
        rsi_period   = trial.suggest_categorical("rsi_period",   [7, 10, 14, 21, 28])
        rebal_freq   = trial.suggest_categorical("rebal_freq",   [1, 2, 4, 6, 8, 12])
        ridge_lambda = trial.suggest_float(      "ridge_lambda", 0.10, 2.0, log=True)
        top_n        = trial.suggest_categorical("top_n",        [3, 5, 7, 9, 12])

        # Compute features for this rsi_period if not already cached
        if rsi_period not in feat_cache:
            feat_cache[rsi_period] = compute_features(
                prices, lookback=252, vol_window=60, rsi_period=rsi_period
            )

        result = run_backtest(
            prices,
            feat_cache[rsi_period],
            regimes,
            rsi_period   = rsi_period,
            rebal_freq   = rebal_freq,
            ridge_lambda = ridge_lambda,
            top_n        = top_n,
            mode         = "val",   # NEVER "test" during tuning
        )

        sharpe = result["sharpe"]
        trial.report(sharpe, step=0)
        return sharpe if np.isfinite(sharpe) else -99.0

    sampler = TPESampler(seed=seed)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=0)
    study   = optuna.create_study(
        direction  = "maximize",
        sampler    = sampler,
        pruner     = pruner,
        study_name = "momentum_dow30",
    )

    print(f"\n{'='*60}")
    print(f"  Optuna TPE — {n_trials} trials on VALIDATION split only")
    print(f"  RSI period   : [7, 10, 14, 21, 28]")
    print(f"  Rebal freq   : [1, 2, 4, 6, 8, 12] weeks")
    print(f"  Ridge lambda : log-uniform [0.10, 2.0]")
    print(f"  Top-N longs  : [3, 5, 7, 9, 12]")
    print(f"  Max weight   : 15% per position (hard cap)")
    print(f"{'='*60}\n")

    def progress_cb(study, trial):
        if trial.number % 10 == 0 or trial.number == 0:
            best = study.best_value if study.best_trial else float("nan")
            v    = trial.value if trial.value is not None else float("nan")
            p    = trial.params
            print(
                f"  Trial {trial.number+1:3d}/{n_trials}"
                f"  RSI={p.get('rsi_period','?'):2}  "
                f"rebal={p.get('rebal_freq','?')}w  "
                f"λ={p.get('ridge_lambda', 0):.3f}  "
                f"N={p.get('top_n','?'):2}  "
                f"Sharpe={v:.3f}  (best={best:.3f})"
            )

    study.optimize(objective, n_trials=n_trials, callbacks=[progress_cb])

    best_params = study.best_params
    print(f"\n{'='*60}")
    print(f"  BEST TRIAL  #{study.best_trial.number + 1}")
    print(f"{'='*60}")
    for k, v in best_params.items():
        print(f"    {k:20s}: {v}")
    print(f"    {'Sharpe (val)':20s}: {study.best_value:.4f}")

    return study, best_params

### Visualization



In [146]:
DARK    = "#0a0c0f"
SURFACE = "#111418"
BORDER  = "#232830"
TEXT    = "#e2e8f0"
MUTED   = "#8896a8"
GREEN   = "#22c55e"
RED     = "#ef4444"
AMBER   = "#f59e0b"
BLUE    = "#60a5fa"
PURPLE  = "#a78bfa"

plt.rcParams.update({
    "figure.facecolor": DARK,    "axes.facecolor":  SURFACE,
    "axes.edgecolor":   BORDER,  "axes.labelcolor": MUTED,
    "xtick.color":      MUTED,   "ytick.color":     MUTED,
    "text.color":       TEXT,    "grid.color":      BORDER,
    "grid.linewidth":   0.5,     "font.family":     "monospace",
    "axes.titlecolor":  TEXT,    "axes.titlesize":  10,
    "axes.titleweight": "bold",
})


def plot_test_performance_summary(
    result:     dict,
    best_params: dict,
    save_path:  str = "results/test_performance_summary.png",
):
    """Summary of test period performance: cumulative performance, drawdown, and regime timeline."""
    fig = plt.figure(figsize=(16, 9), facecolor=DARK) # Adjusted figsize for 3 rows, 1 col
    gs  = gridspec.GridSpec(3, 1, figure=fig, hspace=0.45, left=0.07, right=0.97, top=0.92, bottom=0.07)

    dates   = pd.DatetimeIndex(result["date_index"])
    strat   = np.array(result["strat_curve"])
    bench   = np.array(result["bench_curve"])
    regimes = result["regime_log"]
    reg_colors = {"trending": GREEN, "volatile": AMBER, "crash": RED}

    # Panel 1: cumulative performance
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(dates, strat, color=GREEN, lw=2.0,
             label=f"Strategy   CAGR {result['cagr']:.1f}%", zorder=3)
    ax1.plot(dates, bench, color=BLUE,  lw=1.4, ls="--", alpha=0.7,
             label=f"Benchmark  CAGR {result['bench_cagr']:.1f}%", zorder=2)
    prev_reg, seg_start = regimes[0], dates[0]
    for i in range(1, len(dates)):
        if regimes[i] != prev_reg or i == len(dates) - 1:
            ax1.axvspan(seg_start, dates[i], alpha=0.09,
                        color=reg_colors[prev_reg], zorder=1)
            seg_start = dates[i]
            prev_reg  = regimes[i]
    legend_lines   = ax1.get_lines()
    legend_patches = [
        Patch(facecolor=GREEN, alpha=0.4, label="Trending"),
        Patch(facecolor=AMBER, alpha=0.4, label="Volatile (70% size)"),
        Patch(facecolor=RED,   alpha=0.4, label="Crash → cash"),
    ]
    ax1.legend(handles=[*legend_lines, *legend_patches],
               loc="upper left", fontsize=8.5,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT, ncol=2)
    ax1.set_title("CUMULATIVE PERFORMANCE (TEST PERIOD)  (shaded = regime)", pad=8)
    ax1.set_ylabel("Portfolio value  (base = 100)")
    ax1.grid(True, alpha=0.3)

    # Panel 2: drawdown
    ax2 = fig.add_subplot(gs[1, 0])
    peak_arr = np.maximum.accumulate(strat)
    dd_arr   = (strat - peak_arr) / peak_arr * 100.0
    ax2.fill_between(dates, dd_arr, 0, color=RED, alpha=0.45)
    ax2.plot(dates, dd_arr, color=RED, lw=0.8)
    ax2.set_title(f"DRAWDOWN (TEST PERIOD)  (max {result['max_dd']:.1f}%) -- Sharpe: {result['sharpe']:.4f}")
    ax2.set_ylabel("%")
    ax2.grid(True, alpha=0.3)

    # Panel 3: regime timeline
    ax3 = fig.add_subplot(gs[2, 0]) # Changed from ax4
    reg_int = [{"trending": 1, "volatile": 2, "crash": 3}[r] for r in regimes]
    cmap_r  = {1: GREEN, 2: AMBER, 3: RED}
    for i in range(len(dates) - 1):
        ax3.axvspan(dates[i], dates[i + 1], ymin=0, ymax=1,
                    color=cmap_r[reg_int[i]], alpha=0.75)
    ax3.set_yticks([])
    ax3.set_title(f"REGIME TIMELINE (TEST PERIOD)  (kill-switch fired {result['kills']}×)")
    n = len(regimes)
    ax3.legend(handles=[
        Patch(color=GREEN,
              label=f"Trending  {regimes.count('trending')/n*100:.0f}%"),
        Patch(color=AMBER,
              label=f"Volatile  {regimes.count('volatile')/n*100:.0f}%"),
        Patch(color=RED,
              label=f"Crash     {regimes.count('crash')/n*100:.0f}%"),
    ], loc="upper right", fontsize=8, facecolor=SURFACE,
       edgecolor=BORDER, labelcolor=TEXT)

    fig.suptitle(
        "DOW 30 MOMENTUM STRATEGY  //  TEST PERIOD PERFORMANCE",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.97
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"\n  → Saved: {save_path}")
    plt.close()

In [147]:
def plot_validation_performance(
    result:     dict,
    best_params: dict,
    prices:     pd.DataFrame,
    regimes:    pd.Series,
    save_path:  str = "results/validation_performance.png",
):
    """Summary of validation period performance: cumulative performance and drawdown.
    This is essentially a simplified version of plot_test_performance_summary for the validation period.
    """
    fig = plt.figure(figsize=(16, 6), facecolor=DARK)
    gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.45, left=0.07, right=0.97, top=0.92, bottom=0.07)

    dates   = pd.DatetimeIndex(result["date_index"])
    strat   = np.array(result["strat_curve"])
    bench   = np.array(result["bench_curve"])
    regimes_log = result["regime_log"]
    reg_colors = {"trending": GREEN, "volatile": AMBER, "crash": RED}

    # Panel 1: cumulative performance
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(dates, strat, color=GREEN, lw=2.0,
             label=f"Strategy   CAGR {result['cagr']:.1f}%", zorder=3)
    ax1.plot(dates, bench, color=BLUE,  lw=1.4, ls="--", alpha=0.7,
             label=f"Benchmark  CAGR {result['bench_cagr']:.1f}%", zorder=2)
    prev_reg, seg_start = regimes_log[0], dates[0]
    for i in range(1, len(dates)):
        if regimes_log[i] != prev_reg or i == len(dates) - 1:
            ax1.axvspan(seg_start, dates[i], alpha=0.09,
                        color=reg_colors[prev_reg], zorder=1)
            seg_start = dates[i]
            prev_reg  = regimes_log[i]
    legend_lines   = ax1.get_lines()
    legend_patches = [
        Patch(facecolor=GREEN, alpha=0.4, label="Trending"),
        Patch(facecolor=AMBER, alpha=0.4, label="Volatile (70% size)"),
        Patch(facecolor=RED,   alpha=0.4, label="Crash → cash"),
    ]
    ax1.legend(handles=[*legend_lines, *legend_patches],
               loc="upper left", fontsize=8.5,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT, ncol=2)
    ax1.set_title("CUMULATIVE PERFORMANCE (VALIDATION PERIOD)  (shaded = regime)", pad=8)
    ax1.set_ylabel("Portfolio value  (base = 100)")
    ax1.grid(True, alpha=0.3)

    # Panel 2: drawdown
    ax2 = fig.add_subplot(gs[1, 0])
    peak_arr = np.maximum.accumulate(strat)
    dd_arr   = (strat - peak_arr) / peak_arr * 100.0
    ax2.fill_between(dates, dd_arr, 0, color=RED, alpha=0.45)
    ax2.plot(dates, dd_arr, color=RED, lw=0.8)
    ax2.set_title(f"DRAWDOWN (VALIDATION PERIOD)  (max {result['max_dd']:.1f}%) -- Sharpe: {result['sharpe']:.4f}")
    ax2.set_ylabel("%")
    ax2.grid(True, alpha=0.3)

    fig.suptitle(
        "DOW 30 MOMENTUM STRATEGY  //  VALIDATION PERIOD PERFORMANCE",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.97
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"\n  → Saved: {save_path}")
    plt.close()

In [148]:
def plot_feature_importance_standalone(
    result:     dict,
    best_params: dict,
    save_path:  str = "results/feature_importance.png",
):
    """Plots the feature importance of the XGBoost model."""
    fig, ax = plt.subplots(figsize=(8, 5), facecolor=DARK)

    feat_imp  = result["feat_imp"]
    fi_labels = ["12-1 Mom", "Vol 60d", f"RSI {best_params['rsi_period']}d"]
    vals      = [feat_imp.get(f, 0.0) for f in FEATURE_COLS]
    vals_pct  = np.array(vals) / sum(vals) * 100.0

    bars = ax.barh(fi_labels, vals_pct,
                    color=[GREEN, BLUE, PURPLE], height=0.5)
    for bar, v in zip(bars, vals_pct):
        ax.text(bar.get_width() + 0.5,
                 bar.get_y() + bar.get_height() / 2,
                 f"{v:.1f}%", va="center", fontsize=9, color=TEXT)
    ax.set_title("XGBOOST FEATURE IMPORTANCE (TEST PERIOD)  (gain %)")
    ax.set_xlim(0, max(vals_pct) * 1.3)
    ax.grid(True, alpha=0.3, axis="x")
    ax.set_xlabel("Relative Importance (%)", color=MUTED)

    fig.suptitle(
        "DOW 30 MOMENTUM STRATEGY  //  FEATURE IMPORTANCE",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.97
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"\n  → Saved: {save_path}")
    plt.close()

In [152]:
def plot_rebalance_weights(
    all_weights_log: list,
    save_path:       str = "results/rebalance_weights.png",
    max_legend_tickers: int = 15, # New parameter to limit tickers in legend
):
    """Plots the portfolio weights at each rebalancing point over time using a stacked bar chart."""
    if not all_weights_log:
        print("\n  No rebalance weights to plot (all_weights_log is empty).")
        return

    # Convert weights log to a DataFrame for easier plotting
    df_weights = pd.DataFrame(all_weights_log)
    df_weights["date"] = pd.to_datetime(df_weights["date"])
    df_weights = df_weights.set_index("date")

    # Prepare data for stacked bar chart
    # Each row is a rebalance date, columns are tickers, values are weights
    weights_pivot = pd.DataFrame(index=df_weights.index)
    for _, row in df_weights.iterrows():
        for ticker, weight in row['weights'].items():
            weights_pivot.loc[row.name, ticker] = weight * 100 # Convert to percentage

    # Fill NaN with 0 for tickers not held at a specific rebalance
    weights_pivot = weights_pivot.fillna(0.0)
    print(f"DEBUG: weights_pivot columns after fillna: {weights_pivot.columns.tolist()}")

    # Check if there are any tickers to plot after processing
    num_tickers = len(weights_pivot.columns)
    if num_tickers == 0:
        print("\n  No tickers with non-zero weights after initial processing.")
        return

    # Filter out tickers that never had any weight (columns that are all zeros)
    # This ensures only tickers that actually held a position are considered.
    weights_pivot = weights_pivot.loc[:, (weights_pivot.sum(axis=0) > 0)]
    print(f"DEBUG: weights_pivot columns after sum > 0 filter: {weights_pivot.columns.tolist()}")

    # If after the first filter, there are no tickers with non-zero weights, exit
    if weights_pivot.empty:
        print("\n  No tickers with non-zero weights after initial filtering (all columns removed).")
        return

    # If there are more tickers than max_legend_tickers, keep only the top N by total weight
    if len(weights_pivot.columns) > max_legend_tickers:
        total_weights_per_ticker = weights_pivot.sum(axis=0)
        print(f"DEBUG: total_weights_per_ticker (before nlargest): {total_weights_per_ticker.index.tolist()}")

        # Ensure total_weights_per_ticker is not empty before calling nlargest
        if total_weights_per_ticker.empty:
            print("\n  No total weights found for filtering top tickers (this should not happen if weights_pivot is not empty).")
            return

        top_tickers_to_show = total_weights_per_ticker.nlargest(max_legend_tickers).index.tolist()
        print(f"DEBUG: top_tickers_to_show: {top_tickers_to_show}")

        # Ensure top_tickers_to_show is not empty before slicing
        if not top_tickers_to_show:
            print("\n  Top tickers list is empty after nlargest call.")
            return

        weights_pivot = weights_pivot[top_tickers_to_show]
        print(f"DEBUG: weights_pivot columns after top N filter: {weights_pivot.columns.tolist()}")

    # Re-evaluate num_tickers after potential filtering
    num_tickers = len(weights_pivot.columns)
    if num_tickers == 0:
        print("\n  No tickers to plot after final filtering for legend clarity.")
        return

    # Sort columns by average weight for consistent stacking order (optional, but helps)
    weights_pivot = weights_pivot.iloc[:, weights_pivot.mean().argsort()[::-1]]

    fig, ax = plt.subplots(figsize=(18, 9), facecolor=DARK)

    # Plot as a stacked bar chart
    # Use plt.colormaps.get_cmap for newer matplotlib versions, and sample it to get a list of colors
    cmap = plt.colormaps.get_cmap('tab20')
    colors = [cmap(i) for i in np.linspace(0, 1, num_tickers)] # Sample colors from the colormap

    weights_pivot.plot.bar(stacked=True, ax=ax, color=colors)

    ax.set_title("PORTFOLIO WEIGHTS AT REBALANCE POINTS (TEST PERIOD)")
    ax.set_xlabel("Date", color=MUTED)
    ax.set_ylabel("Weight (%)", color=MUTED)
    ax.legend(title="Tickers", loc="center left", bbox_to_anchor=(1, 0.5),
              fontsize=8, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45) # Rotate x-axis labels for dates

    # Adjust layout to make room for the legend
    plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust right boundary to fit legend

    fig.suptitle(
        "DOW 30 MOMENTUM STRATEGY  //  REBALANCE WEIGHTS",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.97
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"\n  → Saved: {save_path}")
    plt.close()

In [150]:
def plot_optuna(
    study:     optuna.Study,
    save_path: str = "results/optuna_analysis.png",
):
    """4-panel Optuna analysis: history, RSI×rebal heatmap, lambda scatter,
    top-20 bar chart."""
    trials_df = study.trials_dataframe(
        attrs=("number", "value", "params", "state")
    )
    trials_df = trials_df[trials_df["state"] == "COMPLETE"].copy()
    trials_df.rename(columns={"value": "sharpe"}, inplace=True)
    trials_df.sort_values("sharpe", ascending=False, inplace=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=DARK)
    fig.suptitle(
        "OPTUNA HYPERPARAMETER SEARCH  //  objective: Sharpe ratio (val split)",
        fontsize=11, fontweight="bold", color=TEXT, y=0.98
    )

    cmap = LinearSegmentedColormap.from_list("rg", [RED, AMBER, GREEN])
    vmin = trials_df["sharpe"].quantile(0.10)
    vmax = trials_df["sharpe"].quantile(0.90)

    p_rsi   = "params_rsi_period"
    p_rebal = "params_rebal_freq"
    p_n     = "params_top_n"
    p_lam   = "params_ridge_lambda"

    # 1. Optimisation history
    ax = axes[0, 0]
    ax.scatter(trials_df["number"] + 1, trials_df["sharpe"],
               c=trials_df["sharpe"], cmap=cmap, vmin=vmin, vmax=vmax,
               s=25, alpha=0.8, zorder=3)
    running_best = (trials_df.set_index("number")["sharpe"]
                    .sort_index().cummax())
    ax.plot(running_best.index + 1, running_best.values,
            color=GREEN, lw=1.5, ls="--", zorder=4, label="Running best")
    ax.axhline(0.5, color=AMBER, lw=0.8, ls=":", alpha=0.7)
    ax.set_xlabel("Trial number", color=MUTED)
    ax.set_ylabel("Sharpe ratio", color=MUTED)
    ax.set_title("OPTIMISATION HISTORY", color=TEXT)
    ax.legend(fontsize=8, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3)

    # 2. RSI × rebal heatmap
    ax = axes[0, 1]
    if p_rsi in trials_df.columns and p_rebal in trials_df.columns:
        pivot = (trials_df.groupby([p_rsi, p_rebal])["sharpe"]
                 .mean().unstack(fill_value=np.nan))
        im = ax.imshow(pivot.values, cmap=cmap, aspect="auto",
                       vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{c}w" for c in pivot.columns], fontsize=9)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=9)
        ax.set_xlabel("Rebalance frequency", color=MUTED)
        ax.set_ylabel("RSI period", color=MUTED)
        ax.set_title("RSI × REBAL FREQ  (mean Sharpe)", color=TEXT)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                v = pivot.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=7.5,
                            color="black"
                            if v > trials_df["sharpe"].median() else "white")
        plt.colorbar(im, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    # 3. Sharpe by trial, coloured by lambda
    ax = axes[1, 0]
    sc = ax.scatter(
        trials_df["number"] + 1, trials_df["sharpe"],
        c=trials_df[p_lam] if p_lam in trials_df.columns
          else trials_df["sharpe"],
        cmap=cmap, s=30, alpha=0.7, edgecolors="none"
    )
    plt.colorbar(sc, ax=ax, label="Ridge λ").ax.tick_params(labelcolor=MUTED)
    ax.set_xlabel("Trial number", color=MUTED)
    ax.set_ylabel("Sharpe ratio", color=MUTED)
    ax.set_title("SHARPE BY TRIAL  (colour = Ridge λ)", color=TEXT)
    ax.axhline(0.5, color=GREEN, ls="--", lw=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3)

    # 4. Top-20 bar chart
    ax = axes[1, 1]
    top20 = trials_df.head(20).copy()
    bar_colors = [GREEN if s > 0.5 else AMBER if s > 0.3 else RED
                  for s in top20["sharpe"]]
    ax.bar(range(len(top20)), top20["sharpe"], color=bar_colors, width=0.7)
    labels_20 = []
    for _, r in top20.iterrows():
        rsi = int(r.get(p_rsi, 0))
        rb  = int(r.get(p_rebal, 0))
        lam = r.get(p_lam, 0)
        n   = int(r.get(p_n, 0))
        labels_20.append(f"R{rsi} {rb}w\nλ{lam:.2f} N{n}")
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(labels_20, rotation=90, fontsize=6.5)
    ax.set_ylabel("Sharpe ratio", color=MUTED)
    ax.set_title("TOP 20 TRIALS BY SHARPE", color=TEXT)
    ax.axhline(0.5, color=GREEN, ls="--", lw=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  → Saved: {save_path}")
    plt.close()

### MAIN

 Runs the full pipeline in order. Returns all key objects so they are
 available in the notebook scope for diagnostics after the run completes.

 Data flow and bias checkpoints:


```

   prices      → compute_features → features_df.  
                                        │  
                              ┌─────────┴──────────┐    
                         train split           eval splits.   
                         (≤2020-01-01)         val / test   
                              │
                         train_xgboost()
                              │
                         frozen model ──→ run_backtest(mode="val")  ← Optuna
                              │
                              └──────────→ run_backtest(mode="test") ← final metrics
```

 The test split result is computed exactly once, after Optuna finishes.
 It is never used during tuning.

In [153]:
def main():
    print("\n" + "="*60)
    print("  DOW 30 MOMENTUM STRATEGY  —  FULL PIPELINE")
    print("="*60)
    print(f"  Train : start → {TRAIN_CUTOFF.date()}")
    print(f"  Val   : {TRAIN_CUTOFF.date()} → {VAL_CUTOFF.date()}")
    print(f"  Test  : {VAL_CUTOFF.date()} → end  (reported metrics)")

    # Step 1: Download prices
    print("\n[1/5]  Downloading prices from Yahoo Finance ...")
    prices = fetch_prices()
    print(f"       {len(prices)} trading days  ×  {len(prices.columns)} tickers")
    print(f"       {prices.index[0].date()}  →  {prices.index[-1].date()}")

    # Step 2: Regime detection
    print("\n[2/5]  Detecting regimes (PIT equal-weight index) ...")
    regimes = classify_regimes(prices)
    for state in ["trending", "volatile", "crash"]:
        n = (regimes == state).sum()
        print(f"       {state:10s}: {n:4d} days  ({n/len(regimes)*100:.1f}%)")
    crash_days    = regimes[regimes == "crash"]
    volatile_days = regimes[regimes == "volatile"]
    print(f"       Crash days   : {len(crash_days)}"
          f"  first={crash_days.index.min() if len(crash_days) > 0 else 'none'}"
          f"  last={crash_days.index.max()  if len(crash_days) > 0 else 'none'}")
    print(f"       Volatile days: {len(volatile_days)}")
    print("       Expected crash clusters: Q4-2018, Feb-Mar-2020, Jan-Oct-2022")

    # Step 3: Hyperparameter search on validation split only
    print("\n[3/5]  Optuna TPE search (validation split only) ...")
    study, best_params = run_optuna(prices, regimes, n_trials=N_OPTUNA_TRIALS)
    trials_df = study.trials_dataframe()
    trials_df.to_csv("results/optuna_trials.csv", index=False)
    print(f"       All {len(trials_df)} trials → results/optuna_trials.csv")

    # Step 4: Final backtest on held-out test split
    # This is the ONLY time mode="test" is used. Test data was never seen
    # during training or hyperparameter search.
    print("\n[4/5]  Final backtest on held-out TEST split ...")
    feat_df = compute_features(
        prices,
        lookback   = 252,
        vol_window = 60,
        rsi_period = best_params["rsi_period"],
    )

    # Run backtest for the validation period to get its result, needed for plot_validation_performance
    val_result = run_backtest(
        prices, feat_df, regimes,
        rsi_period   = best_params["rsi_period"],
        rebal_freq   = best_params["rebal_freq"],
        ridge_lambda = best_params["ridge_lambda"],
        top_n        = best_params["top_n"],
        mode         = "val",
    )

    # Run backtest for the test period
    test_result = run_backtest(
        prices, feat_df, regimes,
        rsi_period   = best_params["rsi_period"],
        rebal_freq   = best_params["rebal_freq"],
        ridge_lambda = best_params["ridge_lambda"],
        top_n        = best_params["top_n"],
        mode         = "test",   # test split — never seen by Optuna
    )

    print(f"\n       ── Final metrics (test split: {VAL_CUTOFF.date()} → end) ──")
    print(f"       Ann. return  : {test_result['cagr']:.2f}%"
          f"  (benchmark {test_result['bench_cagr']:.2f}%)")
    print(f"       Sharpe ratio : {test_result['sharpe']:.4f}")
    print(f"       Max drawdown : {test_result['max_dd']:.2f}%")
    print(f"       Cumulative   : {test_result['cum_ret']:.1f}%")
    print(f"       Kill-switches: {test_result['kills']}×")

    # Step 5: Charts and outputs
    print("\n[5/5]  Saving charts and results ...")
    plot_test_performance_summary(test_result, best_params, save_path="results/test_performance_summary.png")
    plot_validation_performance(val_result, best_params, prices, regimes, save_path="results/validation_performance.png")
    plot_feature_importance_standalone(test_result, best_params, save_path="results/feature_importance.png")
    plot_rebalance_weights(test_result["all_weights_log"], save_path="results/rebalance_weights.png")
    plot_optuna(study)

    with open("results/best_params.txt", "w") as f:
        f.write("BEST HYPERPARAMETERS  (Optuna TPE — val split)\n")
        f.write("=" * 40 + "\n")
        for k, v in best_params.items():
            f.write(f"{k:20s}: {v}\n")
        f.write("\nPERFORMANCE  (test split)\n")
        f.write("=" * 40 + "\n")
        f.write(f"{'sharpe':20s}: {test_result['sharpe']:.4f}\n")
        f.write(f"{'cagr_%':20s}: {test_result['cagr']:.2f}\n")
        f.write(f"{'bench_cagr_%':20s}: {test_result['bench_cagr']:.2f}\n")
        f.write(f"{'max_drawdown_%':20s}: {test_result['max_dd']:.2f}\n")
        f.write(f"{'cumulative_%':20s}: {test_result['cum_ret']:.2f}\n")
        f.write(f"{'kill_switches':20s}: {test_result['kills']}\n")
    print("       → results/best_params.txt")

    print(f"\n{'='*60}")
    print("  Done. All outputs written to  results/")
    print("="*60 + "\n")

    return test_result, study, prices, regimes, best_params


# ── Run ────────────────────────────────────────────────────────────────────────
test_result, study, prices, regimes, best_params = main()


  DOW 30 MOMENTUM STRATEGY  —  FULL PIPELINE
  Train : start → 2020-01-01
  Val   : 2020-01-01 → 2022-01-01
  Test  : 2022-01-01 → end  (reported metrics)

[1/5]  Downloading prices from Yahoo Finance ...
       2526 trading days  ×  37 tickers
       2016-04-01  →  2026-04-17

[2/5]  Detecting regimes (PIT equal-weight index) ...


[I 2026-04-21 21:41:36,122] A new study created in memory with name: momentum_dow30


       trending  : 1895 days  (75.0%)
       volatile  :  451 days  (17.9%)
       crash     :  180 days  (7.1%)
       Crash days   : 180  first=2018-02-09 00:00:00  last=2025-05-07 00:00:00
       Volatile days: 451
       Expected crash clusters: Q4-2018, Feb-Mar-2020, Jan-Oct-2022

[3/5]  Optuna TPE search (validation split only) ...

  Optuna TPE — 80 trials on VALIDATION split only
  RSI period   : [7, 10, 14, 21, 28]
  Rebal freq   : [1, 2, 4, 6, 8, 12] weeks
  Ridge lambda : log-uniform [0.10, 2.0]
  Top-N longs  : [3, 5, 7, 9, 12]
  Max weight   : 15% per position (hard cap)



[I 2026-04-21 21:41:54,960] Trial 0 finished with value: 0.7504 and parameters: {'rsi_period': 10, 'rebal_freq': 4, 'ridge_lambda': 1.8276027831785728, 'top_n': 3}. Best is trial 0 with value: 0.7504.


  Trial   1/80  RSI=10  rebal=4w  λ=1.828  N= 3  Sharpe=0.750  (best=0.750)


...

  Trial  61/80  RSI= 7  rebal=8w  λ=0.237  N= 7  Sharpe=0.622  (best=0.959)


[I 2026-04-21 21:45:22,370] Trial 61 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.3165918746426355, 'top_n': 7}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:23,914] Trial 62 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.4383461869915029, 'top_n': 7}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:25,454] Trial 63 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.2941079496415971, 'top_n': 7}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:27,007] Trial 64 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.3901215648566945, 'top_n': 7}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:28,513] Trial 65 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.5273052788171563, 'top_n': 7}. Best is trial 38 with 

  Trial  71/80  RSI=28  rebal=2w  λ=0.233  N= 7  Sharpe=0.597  (best=0.959)


[I 2026-04-21 21:45:40,506] Trial 71 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.32357406768278485, 'top_n': 7}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:41,993] Trial 72 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.4051170468532068, 'top_n': 7}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:44,253] Trial 73 finished with value: 0.5513 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.43596513377763935, 'top_n': 3}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:46,620] Trial 74 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.5728523506690394, 'top_n': 7}. Best is trial 38 with value: 0.9591.
[I 2026-04-21 21:45:48,151] Trial 75 finished with value: 0.9591 and parameters: {'rsi_period': 10, 'rebal_freq': 8, 'ridge_lambda': 0.46336427551319964, 'top_n': 7}. Best is trial 38 wi


  BEST TRIAL  #39
    rsi_period          : 10
    rebal_freq          : 8
    ridge_lambda        : 0.3566158376571574
    top_n               : 7
    Sharpe (val)        : 0.9591
       All 80 trials → results/optuna_trials.csv

[4/5]  Final backtest on held-out TEST split ...

       ── Final metrics (test split: 2022-01-01 → end) ──
       Ann. return  : 14.64%  (benchmark 10.94%)
       Sharpe ratio : 0.9360
       Max drawdown : -21.03%
       Cumulative   : 79.1%
       Kill-switches: 0×

[5/5]  Saving charts and results ...

  → Saved: results/test_performance_summary.png

  → Saved: results/validation_performance.png

  → Saved: results/feature_importance.png
DEBUG: weights_pivot columns after fillna: ['CRM', 'MRK', 'PG', 'VZ', 'INTC', 'MCD', 'DOW', 'IBM', 'AXP', 'KO', 'BA', 'MSFT', 'NKE', 'DIS', 'MMM', 'AAPL', 'V', 'CVX', 'JPM', 'WMT', 'CAT', 'HD', 'AMGN', 'TRV', 'UNH', 'CSCO', 'JNJ', 'AMZN', 'NVDA', 'GS', 'HON', 'SHW']
DEBUG: weights_pivot columns after sum > 0 filter: ['CR

## Diagnostic Helpers

In [ ]:
# ── Diagnostic A: performance decomposition ───────────────────────────────────
strat  = np.array(result["strat_curve"])
bench  = np.array(result["bench_curve"])
dates  = pd.DatetimeIndex(result["date_index"])
rets_s = np.diff(strat) / strat[:-1]
rets_b = np.diff(bench) / bench[:-1]

print(f"Strategy  — ann. vol: {rets_s.std()*np.sqrt(252)*100:.1f}%"
      f"  mean daily ret: {rets_s.mean()*252*100:.2f}%")
print(f"Benchmark — ann. vol: {rets_b.std()*np.sqrt(252)*100:.1f}%"
      f"  mean daily ret: {rets_b.mean()*252*100:.2f}%")
print(f"Correlation strat vs bench : {np.corrcoef(rets_s, rets_b)[0,1]:.3f}")
print(f"Weight sum on last rebalance: {sum(result['weights'].values()):.4f}")

df_rets = pd.DataFrame({"strat": rets_s, "bench": rets_b}, index=dates[1:])
annual  = df_rets.resample("YE").apply(lambda x: (1+x).prod()-1) * 100
print("\nYear-by-year returns:")
print(annual.round(1).to_string())

In [ ]:
# ── Diagnostic B: regime sanity check ────────────────────────────────────────
print("\nCrash days by year:")
crash_days = regimes[regimes == "crash"]
print(crash_days.groupby(crash_days.index.year).count())

probe = pd.Timestamp("2020-03-16")
mkt   = prices.mean(axis=1)
lrets = np.log(mkt / mkt.shift(1))
rvol  = lrets.rolling(20).std() * np.sqrt(252)
rpk   = mkt.rolling(60).max()
rdd   = (mkt - rpk) / rpk
print(f"\nOn {probe.date()}:")
print(f"  Vol:    {rvol.loc[probe]:.3f}  (crash threshold 0.28)")
print(f"  DD:     {rdd.loc[probe]:.3f}  (crash threshold -0.13)")
print(f"  Regime: {regimes.loc[probe]}")

In [ ]:
# ── Diagnostic C: weight distribution ────────────────────────────────────────
nonzero  = {t: w for t, w in result["weights"].items() if w > 0.001}
sorted_w = sorted(nonzero.items(), key=lambda x: x[1], reverse=True)
print(f"\nPositions held: {len(nonzero)}")
for t, w in sorted_w:
    bar = "█" * int(w * 200)
    print(f"  {t:6s}  {w*100:5.1f}%  {bar}")
print(f"Top 3 concentration: {sum(w for _,w in sorted_w[:3])*100:.1f}%")
print(f"Top 5 concentration: {sum(w for _,w in sorted_w[:5])*100:.1f}%")

In [ ]:
# ── Diagnostic D: benchmark vs DIA ETF ───────────────────────────────────────
import yfinance as yf
dia        = yf.download("DIA", start="2022-01-01", end="2026-04-18",
                         auto_adjust=True, progress=False)["Close"]
dia_rets   = dia.pct_change().dropna()
annual_dia = dia_rets.resample("YE").apply(lambda x: (1+x).prod()-1) * 100

comparison = pd.DataFrame({
    "strategy":  annual["strat"] if "strat" in annual.columns else annual.iloc[:, 0],
    "benchmark": annual["bench"] if "bench" in annual.columns else annual.iloc[:, 1],
    "DIA_ETF":   annual_dia.reindex(annual.index).squeeze(),
})
print("\nYear-by-year comparison:")
print(comparison.round(1).to_string())

dia_cagr = ((1 + dia_rets.values).prod() ** (252/len(dia_rets)) - 1) * 100
bench_cagr_check = result["bench_cagr"]
print(f"\nStrategy CAGR  : {result['cagr']:.2f}%")
print(f"Benchmark CAGR : {bench_cagr_check:.2f}%  (should be within ~2% of DIA)")
print(f"DIA ETF CAGR   : {dia_cagr:.2f}%")

Results are good, but likely because params were optimized on periods of time that were similar in terms of big macro events. Given the explosive nature of the past 4 years, it makes sense that a momentum based approach would outperform a benchmark.